# Screen Macro — Object Detection Training

Trains a small object-detection model on a dataset captured/labeled by Screen Macro's "Train new model…" wizard (Detect Object action), and exports it to the `.onnx` format the app's `onnx_detector.py` expects.

**Colab (default, manual):**
1. `Runtime` → `Change runtime type` → select a **GPU** (T4 is fine, free tier).
2. `Runtime` → `Run all`.
3. Wait for a "Choose files" button to appear below step 2's cell (can take a minute — it installs dependencies first), click it, and upload the `..._dataset.zip` file Screen Macro produced.
4. Everything else runs on its own — training, export, and an optional sanity check (step 6, safe to skip) — then `best.onnx` downloads automatically at the end. Import it back into the same action via "Import model…", and set its **class filter to 0** (the model's other output class, `1`, is background — see step 5's note).

**Kaggle (opt-in, scripted):** the app's `cv_training.py` pushes this notebook as a Kaggle kernel with the dataset attached as a Kaggle Dataset, polls until it finishes, and pulls `best.onnx` back automatically — no manual steps. The notebook auto-detects which platform it's running on (`ON_KAGGLE`, cell 4) and adjusts the upload/download cells accordingly; everything else runs identically either way.

**Status: verified end-to-end on a real Colab run (2026-08-21)** — training a keypoint-preview model through all 50 epochs and exporting a working `best.onnx` that produces sane detections. The export step (5) needed substantially more rework than originally expected; see its own cell for what changed and why.


## 1. Install dependencies

In [ ]:
!pip install -q "rfdetr[train,loggers]" onnx onnxruntime


## 2. Upload and unpack the dataset

Expects the zip Screen Macro's "Train new model…" wizard produces: an `images/` folder of captured frames plus a `labels.json` of the form
`{"images_dir": "images", "labels": [{"image": "frame_001.png", "objects": [{"box": [x, y, w, h], "keypoint": [x, y]}, ...]}, ...]}`
(frames with no entry in `labels` were skipped during labeling — the object wasn't visible in them; each frame can list more than one object).


In [ ]:
import os, glob, zipfile, shutil
from pathlib import Path

RAW_DIR = Path("dataset_raw")
if RAW_DIR.exists():
    shutil.rmtree(RAW_DIR)

# cv_training.py's push already uploaded the dataset zip as this kernel's one
# attached Kaggle Dataset -- mounted read-only under /kaggle/input, no interactive
# upload prompt needed the way Colab's files.upload() requires. Detected by whether
# a zip is actually there, not just os.path.exists("/kaggle/input") -- that directory
# exists (empty) on plain Colab too, which previously made this always take the
# Kaggle branch there and crash on an empty candidates list.
candidates = glob.glob("/kaggle/input/*/*.zip") + glob.glob("/kaggle/input/*.zip")
ON_KAGGLE = bool(candidates)
if ON_KAGGLE:
    zip_name = candidates[0]
else:
    from google.colab import files
    uploaded = files.upload()
    zip_name = next(iter(uploaded))

with zipfile.ZipFile(zip_name) as zf:
    zf.extractall(RAW_DIR)

print("Extracted:", list(RAW_DIR.iterdir()))


## 3. Convert to COCO format (train/valid split)

One category (`"object"`) with one keypoint (`"click_point"`) per COCO's keypoint-annotation convention — `keypoints: [x, y, visibility]`, `visibility=2` meaning "labeled and visible" (every entry here is, since the labeling step only records a keypoint when the object was actually clicked on). A 90/10 split is a reasonable default for a small, single-object dataset; adjust `VALID_FRACTION` if you captured a lot more images.

In [ ]:
import json, random

VALID_FRACTION = 0.1
COCO_DIR = Path("dataset_coco")

with open(RAW_DIR / "labels.json") as f:
    raw = json.load(f)
images_dir = RAW_DIR / raw.get("images_dir", "images")
entries = raw["labels"]
random.seed(0)
random.shuffle(entries)
n_valid = max(1, int(len(entries) * VALID_FRACTION)) if len(entries) > 1 else 0
splits = {"valid": entries[:n_valid], "train": entries[n_valid:]}

import cv2

for split_name, split_entries in splits.items():
    split_dir = COCO_DIR / split_name
    split_dir.mkdir(parents=True, exist_ok=True)
    images_out, annotations_out = [], []
    for i, e in enumerate(split_entries, start=1):
        src = images_dir / e["image"]
        img = cv2.imread(str(src))
        h, w = img.shape[:2]
        shutil.copy(src, split_dir / e["image"])
        images_out.append({"id": i, "file_name": e["image"], "width": w, "height": h})
        # Each frame can carry more than one labeled object (see the schema note
        # above) -- one COCO annotation per object, all sharing this frame's image_id.
        for obj in e["objects"]:
            bx, by, bw, bh = obj["box"]
            kx, ky = obj["keypoint"]
            annotations_out.append({
                "id": len(annotations_out) + 1, "image_id": i, "category_id": 1,
                "bbox": [bx, by, bw, bh],
                "area": bw * bh,
                "iscrowd": 0,
                "keypoints": [kx, ky, 2],
                "num_keypoints": 1,
            })
    coco = {
        "images": images_out,
        "annotations": annotations_out,
        "categories": [{
            "id": 1, "name": "object", "supercategory": "object",
            "keypoints": ["click_point"], "skeleton": [],
        }],
    }
    with open(split_dir / "_annotations.coco.json", "w") as f:
        json.dump(coco, f)
    print(f"{split_name}: {len(images_out)} image(s), {len(annotations_out)} annotation(s)")


## 4. Train

Tries RF-DETR's keypoint-preview model first (predicts the click point directly, per `docs/cv-object-detection-investigation.md`'s recommendation); falls back to the plain box-only model if the preview class isn't importable or training on it fails — a fully-supported degradation, not an error path (`onnx_detector.py` already handles a keypoint-less `[N,6]` export by clicking the box center instead).

**Per-epoch metrics** (mAP/mAR/F1) print below this cell as each epoch finishes, and are also logged to TensorBoard — run this in a separate cell (before or after starting training) to watch them update live:
```python
%load_ext tensorboard
%tensorboard --logdir output
```

**Early stopping** is enabled below: training stops once the monitored metric (mAP@50:95) hasn't improved by at least `EARLY_STOPPING_MIN_DELTA` for `EARLY_STOPPING_PATIENCE` epochs in a row, instead of always running the full `EPOCHS`. Raise the patience (or set `early_stopping=False` in both `model.train()` calls below) if a small/noisy dataset stops too eagerly.


In [ ]:
KEYPOINT_MODE = True
try:
    from rfdetr import RFDETRKeypointPreview
    model = RFDETRKeypointPreview()
except Exception as exc:
    print("Keypoint-preview model unavailable, falling back to box-only:", exc)
    KEYPOINT_MODE = False

if not KEYPOINT_MODE:
    from rfdetr import RFDETRNano
    model = RFDETRNano()

EPOCHS = 50       # small dataset -- more epochs than a COCO-scale run, adjust if it overfits
BATCH_SIZE = 4    # keep small: free-tier T4 memory headroom is the main constraint here
EARLY_STOPPING_PATIENCE = 10     # stop once mAP@50:95 hasn't improved for this many epochs
EARLY_STOPPING_MIN_DELTA = 0.001 # smallest mAP improvement that still counts as progress

try:
    model.train(
        dataset_dir=str(COCO_DIR), epochs=EPOCHS, batch_size=BATCH_SIZE,
        early_stopping=True, early_stopping_patience=EARLY_STOPPING_PATIENCE,
        early_stopping_min_delta=EARLY_STOPPING_MIN_DELTA,
    )
except Exception as exc:
    if KEYPOINT_MODE:
        print("Keypoint-preview training failed, falling back to box-only:", exc)
        from rfdetr import RFDETRNano
        KEYPOINT_MODE = False
        model = RFDETRNano()
        model.train(
            dataset_dir=str(COCO_DIR), epochs=EPOCHS, batch_size=BATCH_SIZE,
            early_stopping=True, early_stopping_patience=EARLY_STOPPING_PATIENCE,
            early_stopping_min_delta=EARLY_STOPPING_MIN_DELTA,
        )
    else:
        raise

## 5. Export to ONNX with the app's expected contract

`onnx_detector.py` expects a single output tensor per image: `[N, 6]` (x1, y1, x2, y2, confidence, class_id) or `[N, 8]` (…+keypoint_x, keypoint_y) in the model's own fixed input-size pixel coordinates — with NMS/decoding already applied, so the shipped app never needs architecture-specific postprocessing.

This traces the model's own export-mode forward pass (`raw_module.export()` + `forward_export()`) together with its `PostProcess` module directly — both are plain tensor-op code, unlike `predict()`'s heavy multi-input-type handling and `Detections`/`KeyPoints` object construction, which turned out to be untraceable (and, when the raw training-mode forward was traced directly without calling `.export()` first, crashed with a CUDA illegal memory access — multi-scale deformable attention's custom CUDA kernel isn't traceable at all without that step).

Two of `rfdetr`'s own postprocessing internals also needed patching around for export specifically — both scoped to this one `PostProcess` instance, not the whole class, so nothing outside this cell is affected:
- `PostProcess._select_topk` uses `torch.argsort(..., stable=True)`, which the legacy ONNX exporter can't convert at all (`Sort, Out parameter is not supported` — a real version mismatch inside PyTorch itself, not an `rfdetr` bug). Replaced with an equivalent `torch.topk`-based version for the export trace only — same descending-score order, just without the exactly-tied-score tie-break rule, which doesn't matter for this app's use.
- Keypoint-uncertainty score fusion (`trace_alpha`) calls `torch.nextafter`, also unsupported by ONNX. Disabled for export since the app only needs the plain confidence score, not the uncertainty-weighted refinement.

**The exported model's output mixes foreground and background rows** (`class_id=0` is foreground, `class_id=1` is background, per this one-class keypoint setup) — set the Detect Model action's class filter to `0` in the app so it never treats a background row as the best match.


In [ ]:
import torch
import torch.nn as nn
import types

INPUT_SIZE = model.model.resolution

class ExportWrapper(nn.Module):
    """Wraps the raw export-mode LWDETR forward + PostProcess.forward() -- both plain
    tensor ops, no image-decoding/type-dispatch/numpy machinery -- into one flat
    [N,6]/[N,8] tensor per image. Reimplements just the tensor-math half of what
    predict() does; skips predict()'s heavy multi-input-type handling and
    Detections/KeyPoints object construction entirely, which is what made tracing
    predict() itself fail. raw_module.export() must already have been called before
    this wraps it (swaps in export-safe submodule implementations, e.g. multi-scale
    deformable attention's CUDA kernel -- tracing it directly without this step
    crashes with a CUDA illegal memory access)."""

    def __init__(self, raw_module, postprocess_module, keypoint_mode: bool, input_size: int):
        super().__init__()
        self.raw_module = raw_module
        self.postprocess = postprocess_module
        self.keypoint_mode = keypoint_mode
        self.input_size = input_size

    def forward(self, x):
        outputs_coord, outputs_class, *rest = self.raw_module(x)
        out_dict = {"pred_logits": outputs_class, "pred_boxes": outputs_coord}
        if self.keypoint_mode and rest:
            out_dict["pred_keypoints"] = rest[0]
        target_sizes = torch.tensor([[self.input_size, self.input_size]], device=x.device)
        result = self.postprocess(out_dict, target_sizes)[0]  # batch size 1 -> one dict
        boxes = result["boxes"]                     # (num_select, 4) absolute xyxy
        scores = result["scores"].unsqueeze(-1)
        labels = result["labels"].unsqueeze(-1).float()
        cols = [boxes, scores, labels]
        if self.keypoint_mode and "keypoints" in result:
            cols.append(result["keypoints"][:, 0, :2])  # our one "click_point" keypoint's (x, y)
        return torch.cat(cols, dim=-1)


def _select_topk_onnx_friendly(self, out_logits):
    """Replaces PostProcess._select_topk for export only -- see this cell's markdown
    for why its own torch.argsort(..., stable=True) can't be exported."""
    prob = out_logits.sigmoid()
    logits_for_topk = prob.view(out_logits.shape[0], -1)
    num_to_select = min(self.num_select, logits_for_topk.shape[1])
    topk_values, topk_indexes = torch.topk(logits_for_topk, num_to_select, dim=1)
    scores = topk_values
    topk_boxes = topk_indexes // out_logits.shape[2]
    labels = topk_indexes % out_logits.shape[2]
    return scores, labels, topk_boxes


raw_module = model.model.model
raw_module.export()   # swap in export-safe submodule implementations -- see docstring above
raw_module.eval()
for p in raw_module.parameters():
    p.requires_grad = False

# Only patches this one PostProcess instance -- doesn't touch the class or any other
# instance, so normal predict() calls elsewhere in this notebook are unaffected.
model.model.postprocess._select_topk = types.MethodType(_select_topk_onnx_friendly, model.model.postprocess)
# Disables keypoint-uncertainty score fusion, which internally calls torch.nextafter
# (not exportable to ONNX at all) -- we only want the plain confidence score/keypoint
# xy this app actually uses, not the uncertainty-weighted refinement.
model.model.postprocess.trace_alpha = 0.0

wrapper = ExportWrapper(raw_module, model.model.postprocess, KEYPOINT_MODE, INPUT_SIZE)
wrapper.eval()
dummy_input = torch.zeros(1, 3, INPUT_SIZE, INPUT_SIZE, device=next(raw_module.parameters()).device)

with torch.no_grad():
    torch.onnx.export(
        wrapper, dummy_input, "best.onnx",
        input_names=["images"], output_names=["output0"],
        opset_version=17,
        # The newer default exporter (dynamo=True) needs the onnxscript package (not
        # installed here) and can't trace this wrapper's predict()-adjacent code anyway
        # -- see this cell's markdown.
        dynamo=False,
    )
print("Exported best.onnx (keypoint_mode=", KEYPOINT_MODE, ")")


## 6. Sanity-check the export (optional)

Quick check that the exported graph actually produces plausible detections before downloading it — confirms the output shape and that a real training image's top-scoring row looks like a real detection (a sensible box, a confident score, a keypoint that falls inside that box), not noise. Safe to skip (or delete this cell) if you'd rather just download `best.onnx` and test it directly in the app — this only exists to catch a broken export earlier, not because the app needs it.


In [ ]:
import onnxruntime as ort
import numpy as np
from PIL import Image

sess = ort.InferenceSession("best.onnx", providers=["CPUExecutionProvider"])
img = Image.open(next((RAW_DIR / "images").glob("*.png"))).convert("RGB").resize((INPUT_SIZE, INPUT_SIZE))
blob = (np.array(img).astype(np.float32) / 255.0).transpose(2, 0, 1)[None, ...]
output = sess.run(None, {"images": blob})[0]
print("output shape:", output.shape)  # (num_select, 6 or 8): x1,y1,x2,y2,score,label[,kx,ky]
top = output[np.argsort(-output[:, 4])[:5]]
print(top)


## 7. Download the trained model


In [ ]:
if ON_KAGGLE:
    # Kaggle captures every file left in /kaggle/working/ as this kernel's output --
    # no equivalent of Colab's interactive download; cv_training.py's poll/pull step
    # fetches it afterward via `kaggle kernels output`.
    print("On Kaggle: best.onnx left in /kaggle/working/ -- fetched by the app's Kaggle poll/pull step.")
else:
    from google.colab import files
    files.download("best.onnx")